# Train Urgency $E_i$ trên Colab

Chạy các cell theo thứ tự. Notebook tự huấn luyện; không cần upload file `.py`. Dataset được đọc từ Google Drive trong thư mục `NCKH/Ei`. File trên Drive phải là CSV thật có tên `urgency_training_SOURCE_FINAL.csv`, không phải Google Docs/Sheets.

Bộ CSV hiện tại là 32 tổ hợp tổng hợp với nhãn theo quy tắc, không phải dữ liệu ca bệnh có nhãn chuyên gia. Kết quả khớp trên CSV này không phải độ chính xác trên ca thực tế.

In [ ]:
%pip install -q numpy scikit-learn

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import sklearn
from google.colab import drive, files

print(f'Python {sys.version.split()[0]}, numpy {np.__version__}, scikit-learn {sklearn.__version__}')
drive.mount('/content/drive')

dataset_name = 'urgency_training_SOURCE_FINAL.csv'
drive_root = Path('/content/drive/MyDrive')
candidates = [
    drive_root / 'NCKH' / 'Ei' / dataset_name,
    drive_root / 'Colab Notebooks' / 'NCKH' / 'Ei' / dataset_name,
]
dataset_path = next((path for path in candidates if path.is_file()), None)
if dataset_path is None:
    matches = [path for path in drive_root.rglob(dataset_name)
               if path.parent.name == 'Ei' and path.parent.parent.name == 'NCKH']
    if len(matches) != 1:
        raise FileNotFoundError(
            f'Không tìm thấy duy nhất {dataset_name} trong NCKH/Ei trên MyDrive. '
            'Hãy kiểm tra tên file, định dạng CSV và vị trí trong MyDrive.'
        )
    dataset_path = matches[0]
print('Dataset:', dataset_path)

In [ ]:
import csv
from sklearn.linear_model import LogisticRegression

FEATURES = [
    'unresponsive',
    'respiratory_distress_or_cyanosis',
    'heavy_bleeding',
    'active_convulsions',
    'high_risk_trauma',
]
TARGET = 'urgency_label'
with dataset_path.open(encoding='utf-8-sig', newline='') as f:
    rows = list(csv.DictReader(f))
expected_columns = ['profile_id'] + FEATURES + [TARGET]
if not rows or list(rows[0]) != expected_columns:
    raise ValueError(f'CSV phải có đúng các cột theo thứ tự: {expected_columns}')
seen = set()
for row_number, row in enumerate(rows, start=2):
    profile = tuple(int(row[name]) for name in FEATURES)
    if profile in seen:
        raise ValueError(f'Trùng feature profile tại dòng CSV {row_number}: {profile}')
    seen.add(profile)
    for name in FEATURES + [TARGET]:
        if row[name] not in {'0', '1'}:
            raise ValueError(f'Dòng {row_number}: {name} phải là 0/1')
    if int(row[TARGET]) != int(any(profile)):
        raise ValueError(f'Dòng {row_number}: nhãn không khớp quy tắc ánh xạ')
if len(rows) != 32 or len(seen) != 32:
    raise ValueError('CSV phải có đủ 32 tổ hợp feature duy nhất (2^5).')

X = np.array([[int(row[name]) for name in FEATURES] for row in rows], dtype=float)
y = np.array([int(row[TARGET]) for row in rows], dtype=int)
model = LogisticRegression(
    solver='liblinear', C=1.0, class_weight='balanced',
    max_iter=2000, random_state=20260928,
)
model.fit(X, y)
probabilities = model.predict_proba(X)[:, 1]
predictions = (probabilities >= 0.5).astype(int)
artifact = {
    'schema_version': '1.0',
    'model_type': 'logistic_regression',
    'dataset_sha256': hashlib.sha256(dataset_path.read_bytes()).hexdigest(),
    'feature_order': FEATURES,
    'intercept': float(model.intercept_[0]),
    'coefficients': {name: float(value) for name, value in zip(FEATURES, model.coef_[0])},
    'threshold': 0.5,
    'class_weight': 'balanced',
    'training_rows': len(rows),
    'unique_feature_profiles': len(seen),
    'full_protocol_state_fidelity_at_0_5': float((predictions == y).mean()),
    'negative_probability': float(probabilities[y == 0][0]),
    'minimum_positive_probability': float(probabilities[y == 1].min()),
    'output_semantics': 'E_i is a guideline-derived mapped-high-acuity evidence score; not a clinically calibrated probability.',
    'claim_boundary': 'No expert labels; not full WHO/IITT triage; no clinical or field-validity claim.',
}
artifact_path = Path('urgency_logistic_SOURCE_FINAL.json')
artifact_path.write_text(json.dumps(artifact, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(artifact, indent=2, ensure_ascii=False))
assert artifact['dataset_sha256'] == hashlib.sha256(dataset_path.read_bytes()).hexdigest()
assert artifact['training_rows'] == 32 and artifact['unique_feature_profiles'] == 32
print('Artifact và SHA-256 dữ liệu: OK')

In [ ]:
import csv
from sklearn.metrics import classification_report, confusion_matrix

with dataset_path.open(encoding='utf-8-sig', newline='') as f:
    rows = list(csv.DictReader(f))
features = artifact['feature_order']
X = np.array([[int(row[name]) for name in features] for row in rows])
y_true = np.array([int(row['urgency_label']) for row in rows])
weights = np.array([artifact['coefficients'][name] for name in features])
probabilities = 1 / (1 + np.exp(-(X @ weights + artifact['intercept'])))
y_pred = (probabilities >= artifact['threshold']).astype(int)

print('ĐÁNH GIÁ TRÊN CHÍNH 32 PROFILE TỔNG HỢP — KHÔNG PHẢI TEST ĐỘC LẬP')
print('Confusion matrix [nhãn 0, nhãn 1]:')
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
print(classification_report(y_true, y_pred, labels=[0, 1], target_names=['không có dấu hiệu', 'có dấu hiệu'], digits=4, zero_division=0))

In [ ]:
files.download(str(artifact_path))